<a href="https://colab.research.google.com/github/wandb/docs/blob/main/colorblind_safe_palettes_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<img src="http://wandb.me/logo-im-png" width="400" alt="Weights & Biases" />

# Colorblind-Safe Run Palettes in W&B

This notebook demonstrates how to use the new colorblind-safe color palettes in Weights & Biases (W&B) workspaces. These palettes are designed to be accessible to users with various forms of color vision deficiency.

## What's New?

W&B now offers:
- **Two colorblind-safe color palettes** for run visualization
- **Five new continuous color palettes** for metric-based coloring

These palettes improve accessibility and ensure that your experiments are easily distinguishable by all team members.

<div><img /></div>

<img src="http://wandb.me/mini-diagram" width="650" alt="Weights & Biases" />

<div><img /></div>

## Setup

First, let's install and import the necessary libraries:

In [ ]:
!pip install -Uq wandb numpy matplotlib

In [ ]:
import wandb
import numpy as np
import matplotlib.pyplot as plt
import random
from datetime import datetime

In [ ]:
# Log in to W&B
wandb.login()

## Creating Sample Runs with Different Configurations

Let's create multiple runs to demonstrate how the colorblind-safe palettes display different experiments. We'll simulate a hyperparameter search with various configurations.

In [ ]:
# Define project name
PROJECT_NAME = f"colorblind-safe-demo-{datetime.now().strftime('%Y%m%d-%H%M%S')}"

# Hyperparameter configurations to test
configs = [
    {"learning_rate": 0.001, "batch_size": 32, "optimizer": "adam", "dropout": 0.2},
    {"learning_rate": 0.01, "batch_size": 64, "optimizer": "sgd", "dropout": 0.3},
    {"learning_rate": 0.005, "batch_size": 128, "optimizer": "adam", "dropout": 0.1},
    {"learning_rate": 0.001, "batch_size": 32, "optimizer": "rmsprop", "dropout": 0.25},
    {"learning_rate": 0.01, "batch_size": 64, "optimizer": "adam", "dropout": 0.15},
    {"learning_rate": 0.005, "batch_size": 128, "optimizer": "sgd", "dropout": 0.35},
    {"learning_rate": 0.001, "batch_size": 64, "optimizer": "adam", "dropout": 0.2},
    {"learning_rate": 0.01, "batch_size": 32, "optimizer": "rmsprop", "dropout": 0.3},
]

print(f"Creating runs in project: {PROJECT_NAME}")
print(f"Number of runs to create: {len(configs)}")

In [ ]:
# Function to simulate training metrics
def simulate_training(config, num_epochs=20):
    """Simulate training metrics based on configuration."""
    # Base performance influenced by hyperparameters
    base_acc = 0.5
    if config["optimizer"] == "adam":
        base_acc += 0.1
    if config["learning_rate"] == 0.005:
        base_acc += 0.05
    if config["batch_size"] == 64:
        base_acc += 0.03
    
    # Generate metrics
    accuracies = []
    losses = []
    
    for epoch in range(num_epochs):
        # Simulate improving accuracy with some noise
        acc = base_acc + (0.4 * (1 - np.exp(-epoch/5))) + random.uniform(-0.02, 0.02)
        acc = min(0.98, max(0, acc))  # Clamp between 0 and 0.98
        
        # Simulate decreasing loss
        loss = 2.0 * np.exp(-epoch/3) + random.uniform(0, 0.1)
        
        accuracies.append(acc)
        losses.append(loss)
    
    return accuracies, losses

In [ ]:
# Create runs with different configurations
print("Creating runs...")
for i, config in enumerate(configs):
    run_name = f"run_{i}_{config['optimizer']}_lr{config['learning_rate']}"
    
    # Initialize W&B run
    run = wandb.init(
        project=PROJECT_NAME,
        name=run_name,
        config=config,
        reinit=True
    )
    
    # Simulate training
    accuracies, losses = simulate_training(config)
    
    # Log metrics
    for epoch, (acc, loss) in enumerate(zip(accuracies, losses)):
        wandb.log({
            "accuracy": acc,
            "loss": loss,
            "epoch": epoch,
            "val_accuracy": acc + random.uniform(-0.05, 0.05),  # Simulated validation accuracy
            "val_loss": loss + random.uniform(-0.1, 0.1),  # Simulated validation loss
        })
    
    # Log final metrics
    wandb.summary["best_accuracy"] = max(accuracies)
    wandb.summary["best_val_accuracy"] = max(accuracies) + random.uniform(-0.03, 0.03)
    wandb.summary["final_loss"] = losses[-1]
    
    # Finish run
    wandb.finish()
    
    print(f"✓ Created {run_name}")

print(f"\n✅ All runs created successfully!")
print(f"\n🔗 View your project at: https://wandb.ai/{wandb.api.default_entity}/{PROJECT_NAME}")

## How to Enable Colorblind-Safe Palettes

To use the colorblind-safe palettes in your W&B workspace:

1. **Navigate to your project** in the W&B web interface
2. **Click on the Workspace tab** from the project sidebar
3. **Click the Settings icon (⚙️)** in the top right corner
4. **Select "Runs"** from the settings drawer
5. **Find the "Color Palette" section**
6. **Choose one of the two colorblind-safe palettes**

### Available Colorblind-Safe Palettes

W&B provides two carefully designed colorblind-safe palettes:

1. **Colorblind-Safe Palette 1**: Optimized for deuteranopia (red-green color blindness)
2. **Colorblind-Safe Palette 2**: Optimized for protanopia and tritanopia

Both palettes ensure that run colors remain distinguishable for users with various forms of color vision deficiency.

## Demonstrating Color Differences

Let's visualize how different color palettes might appear. Note that the actual colorblind-safe palettes are applied in the W&B web interface, not in this notebook.

In [ ]:
# Example colors that might be used in colorblind-safe palettes
# These are examples - actual W&B palettes are configured in the web interface
colorblind_safe_colors = [
    '#1f77b4',  # Blue
    '#ff7f0e',  # Orange
    '#2ca02c',  # Green
    '#d62728',  # Red
    '#9467bd',  # Purple
    '#8c564b',  # Brown
    '#e377c2',  # Pink
    '#7f7f7f',  # Gray
]

# Create a sample plot showing the colors
fig, ax = plt.subplots(1, 1, figsize=(10, 2))
for i, color in enumerate(colorblind_safe_colors):
    ax.add_patch(plt.Rectangle((i, 0), 1, 1, facecolor=color))
    ax.text(i + 0.5, 0.5, f'Run {i+1}', ha='center', va='center', color='white', fontweight='bold')

ax.set_xlim(0, len(colorblind_safe_colors))
ax.set_ylim(0, 1)
ax.set_xticks([])
ax.set_yticks([])
ax.set_title('Example Colorblind-Safe Palette Colors', fontsize=14, pad=20)
plt.tight_layout()
plt.show()

## Continuous Color Palettes for Metric-Based Coloring

In addition to the discrete colorblind-safe palettes for runs, W&B now offers **five new continuous color palettes** for metric-based coloring. These are particularly useful when using the "Color runs by metric" feature.

### How to Use Continuous Color Palettes

1. Go to **Workspace Settings → Runs → Key-based colors**
2. Select a metric to color by (e.g., "accuracy", "loss")
3. Choose one of the five new continuous palettes
4. Set the number of buckets (2-8)

The continuous palettes provide smooth gradients that remain distinguishable for colorblind users.

In [ ]:
# Example: Creating runs with varying final accuracies for metric-based coloring
print("\nCreating additional runs with varying accuracies for metric-based coloring demo...")

accuracy_targets = [0.65, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95]

for i, target_acc in enumerate(accuracy_targets):
    run = wandb.init(
        project=PROJECT_NAME,
        name=f"accuracy_demo_{target_acc}",
        config={"target_accuracy": target_acc},
        reinit=True
    )
    
    # Log metrics trending toward target accuracy
    for epoch in range(10):
        acc = target_acc * (epoch + 1) / 10 + random.uniform(-0.02, 0.02)
        wandb.log({
            "accuracy": acc,
            "epoch": epoch
        })
    
    wandb.summary["final_accuracy"] = target_acc
    wandb.finish()
    print(f"✓ Created run with target accuracy: {target_acc}")

print("\n✅ Metric-based coloring demo runs created!")

## Benefits of Colorblind-Safe Palettes

### 1. **Improved Accessibility**
- Makes W&B dashboards accessible to the ~8% of men and ~0.5% of women with color vision deficiency
- Ensures all team members can effectively analyze experiments

### 2. **Better Visual Distinction**
- Colors are chosen to maximize contrast and distinguishability
- Works well even on different monitor types and lighting conditions

### 3. **Professional Presentation**
- Suitable for presentations and publications
- Demonstrates commitment to inclusive design

### 4. **Consistent Interpretation**
- All viewers see meaningful distinctions between runs
- Reduces miscommunication about experiment results

## Next Steps

1. **Visit your W&B project** using the link provided above
2. **Navigate to Workspace Settings** and enable a colorblind-safe palette
3. **Experiment with the continuous color palettes** for metric-based coloring
4. **Share your workspace** with confidence that it's accessible to all team members

### Additional Resources

- [W&B Documentation on Run Colors](https://docs.wandb.ai/guides/track/runs/customize-run-colors)
- [W&B Documentation on Metric-Based Coloring](https://docs.wandb.ai/guides/track/runs/color-code-runs)
- [W&B Support](https://wandb.ai/support) for questions and feedback

## Cleanup (Optional)

If you want to delete the demo project after exploring the colorblind-safe palettes:

In [ ]:
# Uncomment the following lines to delete the demo project
# api = wandb.Api()
# project = api.project(f"{wandb.api.default_entity}/{PROJECT_NAME}")
# project.delete()